# Block 03: OF Baseline, CRB, and Real Rank-1 Verification

**Goal**  
Use the real K-alpha dataset to anchor rank-1 equivalence, bridge diagnostics, and observed-vs-predicted amplitude spread.

**Checklist alignment**  
Implements the real-data side of E5 and the full real verification target E12.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [2]:
out = run_block03_real_rank1(cfg)
summary = pd.Series(out["summary"], name="value").to_frame()
display(summary)
display(out["bridge_df"])

/Users/wongdowling/Documents/noise-weighted-subspace-reconstruction/implementation/notebook_support.py:382: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  ks = stats.ks_2samp(resid_a, resid_b)


,value
weighted_cosine_rank1,9.999952e-01
amplitude_correlation,9.999986e-01
median_relative_error,1.049802e-05
rq_alignment_median_abs_diff,9.524219e-01
residual_ks_statistic,1.302083e-01
residual_ks_pvalue,2.946169e-03
sigma_obs,1.091998e+01
sigma_pred,9.902064e-06
sigma_relative_error,9.999991e-01
shapiro_pvalue,8.062553e-12


,k,principal_angle_cosines,principal_angles_deg,residual_mean_a,residual_mean_b,relative_mean_diff,ks_statistic,ks_pvalue
0,1,[0.9999999999999899],[8.144481502960801e-06],5.739404e+15,5.739404e+15,0.000000,0.002604,1.000000e+00
1,2,"[1.0, 0.9999999999999961]","[0.0, 5.0509930079323e-06]",2.374631e+16,5.730506e+15,3.143841,1.000000,4.475847e-230
2,3,"[1.0, 0.9999999999999969, 0.9999999999999962]","[0.0, 4.517745487845102e-06, 4.9783130603797145e-06]",2.384580e+16,5.725113e+15,3.165124,1.000000,4.475847e-230


In [3]:
amp_df = out["amplitude_df"]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(amp_df["of_gls"], amp_df["empca_rank1"], s=10, alpha=0.5)
lo = min(amp_df["of_gls"].min(), amp_df["empca_rank1"].min())
hi = max(amp_df["of_gls"].max(), amp_df["empca_rank1"].max())
ax.plot([lo, hi], [lo, hi], color="black", lw=1.0, ls="--")
ax.set_xlabel("OF amplitude (GLS)")
ax.set_ylabel("EMPCA rank-1 amplitude")
ax.set_title("Real K-alpha: OF vs rank-1 EMPCA")
save_figure(fig, dirs["figures"] / "block_03_real_rank1_scatter.png")
plt.close(fig)

In [4]:
fig, ax = plt.subplots(figsize=(6, 4))
vals = amp_df["of_gls"].to_numpy()
ax.hist(vals, bins=30, density=True, alpha=0.7, label="OF amplitudes")
mu = vals.mean()
sigma = vals.std(ddof=1)
xs = np.linspace(vals.min(), vals.max(), 300)
ax.plot(xs, stats.norm.pdf(xs, loc=mu, scale=sigma), color="black", lw=1.5, label="Gaussian fit")
ax.set_xlabel("amplitude")
ax.set_ylabel("density")
ax.set_title("Real K-alpha amplitude histogram")
ax.legend()
save_figure(fig, dirs["figures"] / "block_03_real_amplitude_histogram.png")
plt.close(fig)

In [5]:
summary_path = dirs["tables"] / "block_03_e12_real_summary.csv"
amp_path = dirs["tables"] / "block_03_real_rank1_amplitudes.csv"
bridge_path = dirs["tables"] / "block_03_real_bridge.csv"
manifest_path = dirs["manifests"] / "block_03_real_summary.json"
save_dataframe(pd.DataFrame([out["summary"]]), summary_path)
save_dataframe(out["amplitude_df"], amp_path)
save_dataframe(out["bridge_df"], bridge_path)
save_json({"summary": out["summary"], "of_stats": out["of_stats"]}, manifest_path)
pd.DataFrame(
    [
        manifest_row("block_03", "real-support", str(summary_path.relative_to(REPO_ROOT)), cfg, out["bundle"]),
        manifest_row("block_03", "real-support", str(bridge_path.relative_to(REPO_ROOT)), cfg, out["bundle"]),
    ]
)

,block_id,regime,output_path,seed,trace_len,pretrigger,data_source,n_events,template_path,psd_path
0,block_03,real-support,results/tables/block_03_e12_real_summary.csv,314159,32768,4000,CAL-kalpha,4358,data/template_K_alpha_tight.npy,data/weight/noise_psd_pink.npy
1,block_03,real-support,results/tables/block_03_real_bridge.csv,314159,32768,4000,CAL-kalpha,4358,data/template_K_alpha_tight.npy,data/weight/noise_psd_pink.npy
